# Random Forest Classifier Training and Evaluation

This notebook trains a RandomForestClassifier using the same preprocessing pipeline from the previous model comparison notebook.

**Model Configuration:**
- n_estimators=300
- max_depth=None
- random_state=42
- n_jobs=-1

**Evaluation Metrics:**
- Accuracy
- Precision
- Recall
- F1-score
- Confusion Matrix
- ROC Curve

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, RocCurveDisplay, classification_report
)
import time

# Set seed for reproducibility
RANDOM_STATE = 42
sns.set_style('whitegrid')

print("✓ All libraries imported successfully!")

In [ ]:
# Record library versions for reproducibility
import sys
import numpy as np
import pandas as pd
import sklearn
import matplotlib
import seaborn as sns

print("="*60)
print("LIBRARY VERSIONS (for reproducibility)")
print("="*60)
print(f"Python version: {sys.version}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"Scikit-learn version: {sklearn.__version__}")
print(f"Matplotlib version: {matplotlib.__version__}")
print(f"Seaborn version: {sns.__version__}")
try:
    print(f"Random State: {RANDOM_STATE}")
except NameError:
    print("Random State: Not defined yet")
print("="*60)

## 1. Load and Prepare Data

Using the same data loading approach as the previous notebook to ensure consistency.

In [ ]:
# Load dataset
df = pd.read_csv('my_data .csv')

# Split features and target
X = df.drop(columns=['placed'])
y = df['placed']

# Identify feature types
num_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_features = X.select_dtypes(include=['object']).columns.tolist()

print(f"Dataset shape: {df.shape}")
print(f"Number of numerical features: {len(num_features)}")
print(f"Number of categorical features: {len(cat_features)}")
print(f"\nNumerical features: {num_features}")
print(f"Categorical features: {cat_features}")
print(f"\nClass distribution:\n{y.value_counts()}")
print(f"\nClass distribution (%):\n{y.value_counts(normalize=True) * 100}")

# Stratified train/test split (25% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

print(f"\nTraining set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")

## 2. Define Preprocessing Pipeline

Using the same preprocessing pipeline as the previous notebook for consistency.

In [ ]:
# Define transformers for numerical features
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Define transformers for categorical features
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine transformers into preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_features),
        ('cat', cat_transformer, cat_features)
    ]
)

print("✓ Preprocessing pipeline created successfully!")

## 3. Define and Train Random Forest Model

Training RandomForestClassifier with specified parameters:
- **n_estimators=300**: Number of trees in the forest
- **max_depth=None**: Trees grow until all leaves are pure or contain min_samples_split samples
- **random_state=42**: For reproducibility
- **n_jobs=-1**: Use all available CPU cores for parallel processing

In [ ]:
# Create Random Forest pipeline
rf_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

# Train the model and measure training time
print("Training Random Forest Classifier...")
print("="*60)
start_time = time.time()

rf_model.fit(X_train, y_train)

training_time = time.time() - start_time
print(f"✓ Model trained successfully!")
print(f"Training time: {training_time:.2f} seconds ({training_time/60:.2f} minutes)")
print("="*60)

## 4. Make Predictions on Test Set

In [ ]:
# Make predictions
print("Making predictions on test set...")
y_pred = rf_model.predict(X_test)
y_proba = rf_model.predict_proba(X_test)[:, 1]

print(f"✓ Predictions completed!")
print(f"Test set size: {len(y_test)}")
print(f"Predictions shape: {y_pred.shape}")

## 5. Compute Evaluation Metrics

Computing accuracy, precision, recall, and F1-score.

In [ ]:
# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

# Create metrics dictionary
metrics = {
    'Accuracy': accuracy,
    'Precision': precision,
    'Recall': recall,
    'F1-Score': f1,
    'AUC': auc
}

# Display metrics
print("="*60)
print("RANDOM FOREST CLASSIFIER - EVALUATION METRICS")
print("="*60)
for metric_name, metric_value in metrics.items():
    print(f"{metric_name:15s}: {metric_value:.4f}")
print("="*60)

# Display classification report
print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Not Placed', 'Placed']))

## 6. Plot Confusion Matrix

In [ ]:
# Calculate confusion matrix
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

# Create confusion matrix plot
plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlGnBu', cbar=True, 
            xticklabels=['Not Placed', 'Placed'],
            yticklabels=['Not Placed', 'Placed'],
            annot_kws={'size': 16, 'weight': 'bold'})

plt.title('Random Forest Classifier - Confusion Matrix', 
          fontsize=16, fontweight='bold', pad=20)
plt.ylabel('Actual', fontsize=13, fontweight='bold')
plt.xlabel('Predicted', fontsize=13, fontweight='bold')

# Add text annotations for clarity
plt.text(0.5, -0.15, f'True Negatives: {tn}  |  False Positives: {fp}  |  False Negatives: {fn}  |  True Positives: {tp}',
         ha='center', va='center', transform=plt.gca().transAxes, fontsize=11)

plt.tight_layout()
plt.savefig('rf_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Confusion matrix saved as 'rf_confusion_matrix.png'")
print(f"\nConfusion Matrix Breakdown:")
print(f"  True Negatives (TN):  {tn:4d}")
print(f"  False Positives (FP): {fp:4d}")
print(f"  False Negatives (FN): {fn:4d}")
print(f"  True Positives (TP):  {tp:4d}")
print(f"  Total Errors:         {fp + fn:4d}")

## 7. Plot ROC Curve

In [ ]:
# Create ROC curve plot
plt.figure(figsize=(10, 8))

# Plot ROC curve
RocCurveDisplay.from_estimator(
    rf_model, X_test, y_test, 
    color='#2E7D32', 
    lw=3,
    name=f'Random Forest (AUC = {auc:.4f})'
)

# Plot diagonal reference line
plt.plot([0, 1], [0, 1], linestyle='--', lw=2, color='gray', 
         alpha=0.6, label='Random Classifier (AUC = 0.5)')

plt.title('Random Forest Classifier - ROC Curve', 
          fontsize=16, fontweight='bold', pad=20)
plt.xlabel('False Positive Rate', fontsize=13, fontweight='bold')
plt.ylabel('True Positive Rate', fontsize=13, fontweight='bold')
plt.legend(loc='lower right', fontsize=12)
plt.grid(alpha=0.3, linestyle='--')

# Add AUC score annotation
plt.text(0.6, 0.2, f'AUC Score: {auc:.4f}', 
         fontsize=14, fontweight='bold',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig('rf_roc_curve.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ ROC curve saved as 'rf_roc_curve.png'")

## 8. Feature Importance Analysis

Random Forest provides feature importance scores that indicate which features contribute most to the predictions.

In [ ]:
# Get feature importances from the trained Random Forest
rf_classifier = rf_model.named_steps['classifier']
feature_importances = rf_classifier.feature_importances_

# Get feature names after preprocessing
preprocessor_fitted = rf_model.named_steps['preprocessor']

# Get feature names for numerical features
num_feature_names = num_features

# Get feature names for categorical features (after one-hot encoding)
cat_feature_names = []
if len(cat_features) > 0:
    onehot_encoder = preprocessor_fitted.named_transformers_['cat'].named_steps['onehot']
    cat_feature_names = onehot_encoder.get_feature_names_out(cat_features).tolist()

# Combine all feature names
all_feature_names = num_feature_names + cat_feature_names

# Create DataFrame for feature importances
feature_importance_df = pd.DataFrame({
    'Feature': all_feature_names,
    'Importance': feature_importances
}).sort_values('Importance', ascending=False)

# Display top 15 features
print("\nTop 15 Most Important Features:")
print("="*60)
print(feature_importance_df.head(15).to_string(index=False))
print("="*60)

# Plot feature importances (top 15)
plt.figure(figsize=(12, 8))
top_features = feature_importance_df.head(15)
sns.barplot(x='Importance', y='Feature', data=top_features, palette='viridis')
plt.title('Top 15 Feature Importances - Random Forest', 
          fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Importance Score', fontsize=13, fontweight='bold')
plt.ylabel('Feature', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('rf_feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Feature importance plot saved as 'rf_feature_importance.png'")

## 9. Return Results Summary

Final summary of all metrics and generated plots.

In [ ]:
# Create comprehensive results dictionary
results = {
    'metrics': metrics,
    'confusion_matrix': {
        'TN': int(tn),
        'FP': int(fp),
        'FN': int(fn),
        'TP': int(tp),
        'total_errors': int(fp + fn)
    },
    'training_time_seconds': round(training_time, 2),
    'model_parameters': {
        'n_estimators': 300,
        'max_depth': None,
        'random_state': 42,
        'n_jobs': -1
    },
    'plots_generated': [
        'rf_confusion_matrix.png',
        'rf_roc_curve.png',
        'rf_feature_importance.png'
    ],
    'top_features': feature_importance_df.head(10).to_dict('records')
}

# Display final summary
print("\n" + "="*80)
print("RANDOM FOREST CLASSIFIER - FINAL SUMMARY")
print("="*80)

print("\n📊 PERFORMANCE METRICS:")
for metric_name, metric_value in metrics.items():
    print(f"   {metric_name:15s}: {metric_value:.4f}")

print("\n🔍 CONFUSION MATRIX:")
print(f"   True Negatives:  {tn}")
print(f"   False Positives: {fp}")
print(f"   False Negatives: {fn}")
print(f"   True Positives:  {tp}")
print(f"   Total Errors:    {fp + fn}")

print("\n⏱️  TRAINING TIME:")
print(f"   {training_time:.2f} seconds ({training_time/60:.2f} minutes)")

print("\n📁 GENERATED PLOTS:")
for plot in results['plots_generated']:
    print(f"   ✓ {plot}")

print("\n" + "="*80)
print("✅ Random Forest model training and evaluation completed successfully!")
print("="*80)

# Return results
results

## Summary

This notebook has successfully:

1. ✅ Loaded the dataset using the same approach as the previous notebook
2. ✅ Applied the same preprocessing pipeline (imputation, scaling, one-hot encoding)
3. ✅ Trained a RandomForestClassifier with specified parameters:
   - n_estimators=300
   - max_depth=None
   - random_state=42
   - n_jobs=-1
4. ✅ Made predictions on X_test
5. ✅ Computed evaluation metrics:
   - Accuracy
   - Precision
   - Recall
   - F1-score
   - AUC
6. ✅ Generated visualizations:
   - Confusion Matrix
   - ROC Curve
   - Feature Importance (bonus)
7. ✅ Returned a comprehensive metrics dictionary and plots

### Connection to Previous Notebook

This notebook is connected to `model_evaluation_comparison.ipynb` by:
- Using the **same dataset** (`my_data .csv`)
- Using the **same train/test split** (25% test, stratified, random_state=42)
- Using the **same preprocessing pipeline** (SimpleImputer + StandardScaler for numerical, SimpleImputer + OneHotEncoder for categorical)
- Using the **same evaluation metrics** for fair comparison

This ensures that the Random Forest results can be directly compared with the SVM and KNN models from the previous notebook.